# Text → Knowledge Graph → Neo4j (demo)

A minimal, reproducible walkthrough of building a knowledge graph from unstructured text
with an LLM, loading it into **Neo4j Aura**, and querying it back with Cypher.

**Pipeline:** raw text → LLM structured distillation → entity/relation extraction (`iText2KG`)
→ load into Neo4j → query with Cypher → visualize.

**Stack:** `itext2kg==0.0.3`, `neo4j>=5`, an OpenAI API key, and a free
[Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance.

> Credentials are read from environment variables (or prompted) — nothing is hardcoded.
> Verified against this repo's `.venv` (imports + signatures).


## 0. Install dependencies

Run once, then restart the kernel if needed.


In [ ]:
# %pip install -r requirements_kg_demo.txt


## 1. Configure credentials

Set these as environment variables before launching Jupyter, or you'll be prompted here.

For **Aura**, the connection URI looks like `neo4j+s://<dbid>.databases.neo4j.io`,
the username, and the password is shown once when you create the instance.


In [ ]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY / NEO4J_* from a local .env if present

def need(var, prompt, secret=True):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(prompt) if secret else input(prompt)
    return os.environ[var]

OPENAI_API_KEY = need('OPENAI_API_KEY', 'OpenAI API key: ')
NEO4J_URI      = need('NEO4J_URI', 'Neo4j URI (e.g. neo4j+s://xxxx.databases.neo4j.io): ', secret=False)
NEO4J_USERNAME = os.environ.get('NEO4J_USERNAME', 'neo4j')
NEO4J_PASSWORD = need('NEO4J_PASSWORD', 'Neo4j password: ')
print('Config loaded. URI =', NEO4J_URI)


## 2. Provide the source text

Use the inline sample below so the notebook runs end-to-end out of the box,
or swap in a PDF by uncommenting the `PyPDFLoader` block.


In [ ]:
# sample_text = """
# Marie Curie was a physicist and chemist born in Warsaw, Poland, in 1867.
# She conducted pioneering research on radioactivity, a term she coined.
# In 1903 she shared the Nobel Prize in Physics with her husband Pierre Curie
# and Henri Becquerel. In 1911 she won the Nobel Prize in Chemistry for her
# discovery of the elements polonium and radium. She founded the Curie Institute
# in Paris and is the only person to win Nobel Prizes in two different sciences.
# """

# --- Optional: load a PDF instead ---
from langchain_community.document_loaders import PyPDFLoader
pages = PyPDFLoader('data/your_paper.pdf').load_and_split()
sample_text = '\n'.join(p.page_content for p in pages[:5])

print(sample_text[:200], '...')


## 3. Distill the document

`DocumentsDisiller` uses the LLM to turn free text into a structured summary
(an `Article` schema: title, abstract, key findings, ...), which makes the
downstream entity/relation extraction cleaner.


In [ ]:
from itext2kg.documents_distiller import DocumentsDisiller, Article

IE_query = '''
# DIRECTIVES:
- Act like an experienced information extractor.
- You are given a chunk of text.
- If you do not find the right information, leave its place empty.
'''

distiller = DocumentsDisiller(
    openai_api_key=OPENAI_API_KEY, model_name='gpt-4o', temperature=0
)
distilled = distiller.distill(
    documents=[sample_text.replace('{', '[').replace('}', ']')],
    output_data_structure=Article,
    IE_query=IE_query,
)
distilled


## 4. Build the knowledge graph

`iText2KG.build_graph` extracts entities and the relations between them
from the distilled semantic blocks, returning typed nodes and relationships.


In [ ]:
from itext2kg.graph_integration import iText2KG

semantic_blocks = [f'{k} - {v}'.replace('{', '[').replace('}', ']')
                   for k, v in distilled.items()]

builder = iText2KG(
    openai_api_key=OPENAI_API_KEY,
    model_name='gpt-4o',
    embeddings_model_name='text-embedding-3-large',
    temperature=0,
)
global_ent, global_rel = builder.build_graph(sections=semantic_blocks)

print(f'{len(global_ent)} entities, {len(global_rel)} relationships')


## 5. Load the graph into Neo4j Aura

`GraphIntegrator` writes the nodes and relationships into Neo4j and renders a preview.
After this runs you can also see the graph in the Aura **Query** browser.


In [ ]:
from itext2kg.graph_integration import GraphIntegrator

json_graph = {'nodes': global_ent, 'relationships': global_rel}

GraphIntegrator(
    uri=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
).visualize_graph(json_graph=json_graph)


## 6. Query Neo4j with Cypher

Connect with the official driver and read the graph back


In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def fetch_triples(tx, limit=50):
    q = '''
    MATCH (a)-[r]->(b)
    RETURN coalesce(a.name, elementId(a)) AS source,
           type(r)                        AS relation,
           coalesce(b.name, elementId(b)) AS target
    LIMIT $limit
    '''
    return [record.data() for record in tx.run(q, limit=limit)]

with driver.session() as session:
    triples = session.execute_read(fetch_triples)

for t in triples:
    print(f"({t['source']}) -[{t['relation']}]-> ({t['target']})")


## 7. Visualize the queried graph locally

Build a `networkx` graph from the Cypher results and draw it — handy for slides
without sharing live Aura access.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for t in triples:
    G.add_edge(t['source'], t['target'], label=t['relation'])

# Pass an explicit `ax` so nx.draw skips the cf._axstack() path
# (networkx 2.5.x + matplotlib 3.7.x incompatibility otherwise).
fig, ax = plt.subplots(figsize=(11, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, ax=ax, with_labels=True, node_color='#9ecae1', node_size=1800,
        font_size=8, arrows=True)
nx.draw_networkx_edge_labels(G, pos, ax=ax,
        edge_labels=nx.get_edge_attributes(G, 'label'), font_size=7)
ax.set_title('Knowledge graph queried from Neo4j')
ax.axis('off')
plt.show()

driver.close()


---
### Notes & troubleshooting
- **Pin versions:** `itext2kg==0.0.3` takes `openai_api_key`/`model_name` directly.
  Newer releases switch to LangChain model objects (`llm_model=`) and rename things — upgrade deliberately.
- **Aura URI scheme:** use `neo4j+s://` (TLS). Local Neo4j uses `bolt://localhost:7687`.
- **Cost:** distillation + extraction call GPT-4o; a short text is a few cents.
- **Empty graph?** Try a longer, fact-dense source text — sparse text yields few relations.
- **`Article` schema** is paper-oriented (title/abstract/key_findings/...). For non-paper text it still
  works, but you can define your own pydantic schema and pass it as `output_data_structure`.
